<a href="https://colab.research.google.com/github/rodrigologin0-cpu/Rodrigo-de-Souza-Lima/blob/main/NARX_%2B_NSGA_II_COMPLETO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# NARX + NSGA-II COMPLETO
# Rodrigo Lima - versão publicação
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random, warnings
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")
np.random.seed(42)
random.seed(42)

# ============================================================
# CONFIGURAÇÕES
# ============================================================

ARQUIVO = "Dataset.xlsx"

lag = 3
degree = 2
horizons = list(range(1, 11))

pop_size = 20
generations = 10
mutation_rate = 0.1
initial_prob = 0.1
max_terms = 15

ACCEPTABLE_ERROR = 10

# ============================================================
# 1. LEITURA + CONVERSÃO SEG → MIN
# ============================================================

df_sec = pd.read_excel(ARQUIVO).dropna().reset_index(drop=True)
df_sec = df_sec.select_dtypes(include=[np.number])

# Média a cada 60 segundos
df = df_sec.groupby(df_sec.index // 60).mean().reset_index(drop=True)

col_y = df.columns[0]
cols_u = list(df.columns[1:])

# ============================================================
# FUNÇÕES
# ============================================================

def build_dataset(H):
    data = pd.DataFrame(index=df.index)

    data[f"{col_y}_t"] = df[col_y]
    for i in range(1, lag+1):
        data[f"{col_y}_t-{i}"] = df[col_y].shift(i)

    for c in cols_u:
        data[f"{c}_t"] = df[c]
        for i in range(1, lag+1):
            data[f"{c}_t-{i}"] = df[c].shift(i)

    data["target"] = df[col_y].shift(-H)
    data = data.dropna().reset_index(drop=True)

    X = data.drop(columns=["target"])
    y = data["target"].values

    split = int(0.8 * len(X))

    scaler = StandardScaler()
    Xtr = scaler.fit_transform(X.iloc[:split])
    Xte = scaler.transform(X.iloc[split:])

    poly = PolynomialFeatures(degree=degree, include_bias=False)
    Xtr = poly.fit_transform(Xtr)
    Xte = poly.transform(Xte)

    names = poly.get_feature_names_out(X.columns)

    return Xtr, Xte, y[:split], y[split:], names

# ------------------------------------------------------------

def metrics(y, yp):
    err = y - yp
    return {
        "RMSE": np.sqrt(mean_squared_error(y, yp)),
        "MAE": mean_absolute_error(y, yp),
        "R2": r2_score(y, yp),
        "ROB": np.std(err),
        "PCT_10": 100*np.mean(np.abs(err) <= ACCEPTABLE_ERROR)
    }

# ------------------------------------------------------------

def cost(mask, names):
    c = 0
    for i, m in enumerate(mask):
        if m:
            term = names[i]
            if " " in term or "^" in term:
                c += 4
            else:
                c += 2
    return c

# ------------------------------------------------------------

def evaluate(mask, Xtr, Xte, ytr, yte, names):
    mask = mask.copy()

    if mask.sum() == 0:
        mask[np.random.randint(len(mask))] = True

    while mask.sum() > max_terms:
        idx = np.where(mask)[0]
        mask[np.random.choice(idx)] = False

    model = LinearRegression()
    model.fit(Xtr[:, mask], ytr)
    yp = model.predict(Xte[:, mask])

    m = metrics(yte, yp)

    return {
        "mask": mask,
        "pred": yp,
        "RMSE": m["RMSE"],
        "COST": cost(mask, names),
        "ROB": m["ROB"]
    }

# ------------------------------------------------------------

def dominates(a, b):
    return (
        a["RMSE"] <= b["RMSE"] and
        a["COST"] <= b["COST"] and
        a["ROB"] <= b["ROB"]
    ) and (
        a["RMSE"] < b["RMSE"] or
        a["COST"] < b["COST"] or
        a["ROB"] < b["ROB"]
    )

# ------------------------------------------------------------

def pareto(pop):
    return [p for p in pop if not any(dominates(q, p) for q in pop if q is not p)]

# ------------------------------------------------------------

def nsga(Xtr, Xte, ytr, yte, names):
    n = Xtr.shape[1]

    pop = []
    for _ in range(pop_size):
        mask = np.random.rand(n) < initial_prob
        pop.append(evaluate(mask, Xtr, Xte, ytr, yte, names))

    for _ in range(generations):
        pf = pareto(pop)
        offspring = []

        while len(offspring) < pop_size:
            parent = random.choice(pf)
            mask = parent["mask"].copy()

            for i in range(len(mask)):
                if random.random() < mutation_rate:
                    mask[i] = not mask[i]

            offspring.append(evaluate(mask, Xtr, Xte, ytr, yte, names))

        pop = pareto(pop + offspring)

    pf = pareto(pop)

    # Escolha do melhor compromisso
    best = min(pf, key=lambda x: x["RMSE"] + x["COST"]/100 + x["ROB"])

    return best, pf

# ============================================================
# EXECUÇÃO
# ============================================================

results = []

for H in horizons:

    Xtr, Xte, ytr, yte, names = build_dataset(H)

    # FULL
    full = LinearRegression().fit(Xtr, ytr)
    pred_full = full.predict(Xte)
    m_full = metrics(yte, pred_full)

    # LASSO
    lasso = Lasso(alpha=0.05).fit(Xtr, ytr)
    pred_lasso = lasso.predict(Xte)
    m_lasso = metrics(yte, pred_lasso)

    # NSGA
    nsga_model, pf = nsga(Xtr, Xte, ytr, yte, names)
    pred_nsga = nsga_model["pred"]

    results.append({
        "H": H,
        "Full_RMSE": m_full["RMSE"],
        "Lasso_RMSE": m_lasso["RMSE"],
        "NSGA_RMSE": nsga_model["RMSE"],
        "NSGA_COST": nsga_model["COST"]
    })

df_res = pd.DataFrame(results)

# ============================================================
# GRÁFICOS
# ============================================================

# 1. Degradação horizonte
plt.figure(figsize=(10,5))
plt.axhspan(0,10,alpha=0.2,label="±10°C")
plt.plot(df_res["H"], df_res["Full_RMSE"], label="Full")
plt.plot(df_res["H"], df_res["Lasso_RMSE"], label="Lasso")
plt.plot(df_res["H"], df_res["NSGA_RMSE"], label="NSGA")
plt.legend()
plt.title("Degradação do horizonte")
plt.show()

# 2. Erro vs custo
plt.figure(figsize=(10,5))
plt.scatter(df_res["NSGA_COST"], df_res["NSGA_RMSE"])
plt.axhspan(0,10,alpha=0.2)
plt.title("Erro vs custo")
plt.show()

# 3. Pareto 3D
pf = pf
rmse = [p["RMSE"] for p in pf]
costs = [p["COST"] for p in pf]
rob = [p["ROB"] for p in pf]

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.scatter(costs, rmse, rob)
ax.set_xlabel("Custo")
ax.set_ylabel("RMSE")
ax.set_zlabel("Robustez")
plt.show()